# 04 — End-to-end demo: od `git clone` do survival dataset

Demonstracja **samowystarczalności pipeline'u**. Notebook pokazuje pełną drogę od pustego repo do gotowego do uczenia maszynowego datasetu - bez żadnego ręcznego pobierania danych z portalu GDC, bez ręcznego klikania w przeglądarce.

Krok po kroku:

1. **`luad-huba download`** - pobiera 10 testowych plików z GDC (STAR-Counts, sample_sheet, clinical, metadata.cart.json)
2. **`luad-huba parse-star`** - parsuje pliki STAR-Counts do formatu parquet
3. **`luad-huba validate-cohort`** - kontrola jakości kohorty (raport JSON)
4. **`luad-huba build-matrix`** - buduje macierz ekspresji (z konfiguracją YAML)
5. **`luad-huba build-survival`** - buduje finalny survival dataset (z konfiguracją YAML)
6. Sanity check finalnego datasetu

**Notebook pobiera 10 plików (~40 MB) do katalogu tymczasowego** - nie zaśmieca `data/raw/`, bezpieczny do wielokrotnego odpalania. Dla pełnej kohorty wystarczy zwiększyć `SIZE = 601` lub usunąć `--size`.

**Cel notebooka:** dowód że pipeline jest naprawdę samowystarczalny. Jeśli wszystkie 5 kroków przejdzie bez błędu, pipeline jest gotowy do produkcyjnego użycia na pełnej kohorcie.


In [ ]:
import subprocess
import sys
import tempfile
import shutil
from pathlib import Path
from datetime import datetime, timezone

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

DEMO_DIR = Path(tempfile.mkdtemp(prefix="luad_huba_demo_"))
DATA_RAW = DEMO_DIR / "raw"
DATA_INTERIM = DEMO_DIR / "interim" / "star_counts"
DATA_PROCESSED = DEMO_DIR / "processed"
LOGS_QC = DEMO_DIR / "logs" / "qc"

SIZE = 10

print(f"Projekt: {PROJECT_ROOT}")
print(f"Katalog demo (tymczasowy): {DEMO_DIR}")
print(f"Polars: {pl.__version__}")
print(f"Czas: {datetime.now(timezone.utc).isoformat()}")
print(f"Plików do pobrania w tym demo: {SIZE}")


## Krok 1 — pobranie kohorty z GDC

`luad-huba download` jest pojedynczą komendą, która spina:
- zapytanie do `/files` o metadane plików STAR
- zapis `gdc_sample_sheet.tsv` w formacie portalu
- zapis `metadata.cart.json` z pełnymi metadanymi
- zapytanie do `/cases` o dane kliniczne
- zapis `clinical.tsv`
- pobranie samych plików STAR z weryfikacją MD5

Wynik: 4 typy plików, dokładnie te same które dostalibyśmy z UI portalu.


In [ ]:
def run_cli(args, cwd=PROJECT_ROOT):
    """Uruchamia komendę luad-huba i wypisuje jej output. Rzuca CalledProcessError przy błędzie."""
    print(f"$ python -m src.cli {' '.join(str(a) for a in args)}")
    result = subprocess.run(
        [sys.executable, "-m", "src.cli", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Komenda zakończona z kodem {result.returncode}")
    return result


run_cli([
    "download",
    "--output-dir", str(DATA_RAW),
    "--size", str(SIZE),
])


In [ ]:
# Co rzeczywiście wpadło do DATA_RAW
print("=== Pliki w data/raw/ po download ===")
for path in sorted(DATA_RAW.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"  {path.name:60} {size_mb:>8.2f} MB")


## Krok 2 — parsowanie STAR-Counts

Każdy plik TSV jest przekształcany do parquet (kolumnowy format, szybsze wczytywanie, mniejsze pliki).


In [ ]:
run_cli([
    "parse-star",
    "--input-dir", str(DATA_RAW),
    "--output-dir", str(DATA_INTERIM),
])

print()
print(f"=== Liczba parquetów: {len(list(DATA_INTERIM.glob('*.parquet')))} ===")


## Krok 3 — walidacja kohorty

Cztery reguły QC sprawdzające spójność: brak plików, orphan files, brak clinical, duplikaty próbek. Wynik to JSON ze stemplem czasowym UTC.


In [ ]:
run_cli([
    "validate-cohort",
    "--sample-sheet", str(DATA_RAW / "gdc_sample_sheet.tsv"),
    "--clinical", str(DATA_RAW / "clinical.tsv"),
    "--interim-dir", str(DATA_INTERIM),
    "--log-dir", str(LOGS_QC),
])

reports = sorted(LOGS_QC.glob("qc_report_*.json"))
print()
print(f"Raport QC zapisany: {reports[-1] if reports else 'BRAK'}")


## Krok 4 — budowa macierzy ekspresji

Tu używamy **flagi `--config`** żeby pokazać że konfiguracja YAML faktycznie działa. `configs/default.yaml` mówi `normalization.method: tpm` - macierz zostanie zbudowana z metryki TPM zamiast raw counts.


In [ ]:
run_cli([
    "build-matrix",
    "--input-dir", str(DATA_INTERIM),
    "--sample-sheet", str(DATA_RAW / "gdc_sample_sheet.tsv"),
    "--output-dir", str(DATA_PROCESSED),
    "--config", str(PROJECT_ROOT / "configs" / "default.yaml"),
    "--duplicate-strategy", "deepest",
])

matrix_path = DATA_PROCESSED / "expression_matrix.parquet"
print()
print(f"=== Macierz zapisana: {matrix_path} ===")
print(f"Rozmiar: {matrix_path.stat().st_size / 1024**2:.2f} MB")


## Krok 5 — budowa survival dataset

Druga komenda używająca konfiguracji YAML. Tu działa `survival.min_follow_up_days: 30` - próbki o czasie obserwacji krótszym niż 30 dni zostaną odfiltrowane.


In [ ]:
run_cli([
    "build-survival",
    "--matrix", str(DATA_PROCESSED / "expression_matrix.parquet"),
    "--sample-sheet", str(DATA_RAW / "gdc_sample_sheet.tsv"),
    "--clinical", str(DATA_RAW / "clinical.tsv"),
    "--output-dir", str(DATA_PROCESSED),
    "--config", str(PROJECT_ROOT / "configs" / "default.yaml"),
])

survival_path = DATA_PROCESSED / "survival_dataset.parquet"
print()
print(f"=== Survival dataset zapisany: {survival_path} ===")
print(f"Rozmiar: {survival_path.stat().st_size / 1024**2:.2f} MB")


## Krok 6 — sanity check finalnego datasetu

Czy wynik ma sens? Wczytujemy parquet, sprawdzamy rozmiar, kolumny meta, rozkład event/censoring.


In [ ]:
dataset = pl.read_parquet(survival_path)
print(f"=== Dataset: {dataset.height} próbek x {dataset.width} kolumn ===")

n_metadata = len([c for c in dataset.columns if not c.startswith("ENSG")])
n_genes = len([c for c in dataset.columns if c.startswith("ENSG")])
print(f"  Kolumny metadanych: {n_metadata}")
print(f"  Kolumny genów: {n_genes}")
print()

print("Pierwsze 5 kolumn metadanych:")
print(dataset.select(dataset.columns[:n_metadata]).head(3))
print()

print("Rozkład event/censoring:")
print(dataset.group_by("event").len())


In [ ]:
print("=== Demo zakończone pomyślnie ===")
print()
print(f"Wszystkie artefakty pipeline'u znajdują się w: {DEMO_DIR}")
print()
print("Co właśnie zostało udowodnione:")
print("1. luad-huba download pobiera kohorte z GDC bez żadnej autoryzacji")
print("2. luad-huba parse-star transformuje TSV -> parquet (szybsze IO)")
print("3. luad-huba validate-cohort produkuje raport QC w JSON")
print("4. luad-huba build-matrix --config buduje macierz wg konfiguracji YAML")
print("5. luad-huba build-survival --config produkuje ML-ready dataset")
print()
print("Pipeline jest naprawdę samowystarczalny - bez ręcznych kroków,")
print("bez ściągania niczego z przeglądarki, bez kopiowania plików.")
print()
print(f"Katalog tymczasowy {DEMO_DIR} można usunąć w dowolnym momencie:")
print(f"  shutil.rmtree({DEMO_DIR!r})")


## Wnioski

Pełen pipeline LUAD-HUBA, od pustego repo do gotowego survival dataset, wymaga:

```bash
git clone https://github.com/WorthySubset151/luad-huba-clean.git
cd luad-huba-clean
uv sync
luad-huba download
luad-huba parse-star
luad-huba validate-cohort
luad-huba build-matrix --config configs/default.yaml
luad-huba build-survival --config configs/default.yaml
```

Siedem komend, zero kliknięć w przeglądarce, zero ręcznego kopiowania plików. **Repo jest samowystarczalne**.

Co dalej (poza zakresem tego notebooka):

- Notebook `05_baseline_survival` - pierwsze modele Kaplan-Meier + Cox na survival_dataset.parquet
- Strategia feature selection (60660 genów × 533 próbki → top variable / LASSO / panel ekspercki)
- Warstwa modelowania (`src/models/`)
- Interfejs Streamlit (`src/app/`) - po zbudowaniu modeli
